In [1]:
import sys

sys.path.insert(0, '../../')

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import yaml
from jax import vmap

from iactrace import MCIntegrator, Telescope
from iactrace.viz import show_telescope

# Some useful code:

In [2]:
def spherical_surface(radius, points):
    Z = radius-np.sqrt(radius**2-(points[...,0]**2+points[...,1]**2))
    N = np.stack([-points[...,0], -points[...,1], radius - Z], axis=-1)
    return np.stack([points[...,0], points[...,1], Z], axis=-1), N/radius

def look_at_euler(mirror_pos, target_pos, up=None):
    """
    Compute euler angles to look from mirror_pos towards target_pos.

    Args:
        mirror_pos: Position to look from (3,)
        target_pos: Position to look at (3,)
        up: Up vector (3,)

    Returns:
        Euler angles (3, )
    """
    if up is None:
        up = jnp.array([0., 1., 0.])
    forward = target_pos - mirror_pos
    forward = forward / jnp.linalg.norm(forward)

    right = jnp.cross(up, forward)
    right = right / jnp.linalg.norm(right)

    up_corrected = jnp.cross(forward, right)

    # Rotation matrix: local -> world
    R = jnp.column_stack([right, up_corrected, forward])

    # Compute tilt
    sy = -R[2,0]  # sin(tilt)
    cy = jnp.sqrt(1 - sy**2)

    # Avoid gimbal problem
    tip = jax.lax.cond(
        cy > 1e-6,
        lambda _: jnp.arctan2(R[2,1], R[2,2]),
        lambda _: 0.0,
        operand=None
    )
    rotation = jax.lax.cond(
        cy > 1e-6,
        lambda _: jnp.arctan2(R[1,0], R[0,0]),
        lambda _: jnp.arctan2(-R[0,1], R[1,1]),
        operand=None
    )
    tilt = jnp.arcsin(sy)

    return jnp.rad2deg(jnp.array([tip, tilt, rotation]))

# Loading config files:

Loading files:

In [3]:
# Position of hexagon pixels in NectarCAM:
nectpix = np.load('nect_pix.npy')
flashpix = np.load('flash_pix.npy')

In [4]:
# Positions of mirrors:
df = pd.read_csv('mirror_CTA-100_1.20-86-0.04.dat', comment='#', sep=r'\s+')
df = df.apply(lambda x: x.astype(str).str.replace(',', ''))
df = df.astype(float)

mirrors_xyd = np.array(df[['x', 'y', 'd']].values)/100

# MST North Like:

In [5]:
# Build config dictionary directly
config = {
    "telescope": {"name": "MST_North", "units": "m"},
    "mirror_templates": {
        "spherical_32m": {
            "surface": {
                "curvature": 1/32.14,
                "conic": 0.0,
                "aspheric": [],
            }
        }
    },
    "mirrors": [],
    "obstructions": [],
    "sensors": [],
}

# Focal length
f = 16
R = 19.2

# NectarCam sensor
config["sensors"].append({
    "id": "NectarCAM",
    "type": "hexagonal",
    "position": [0.0, 0.0, f],
    "orientation": [0.0, 0.0, 0.0],
    "centers_x": nectpix[:,0].tolist(),
    "centers_y": nectpix[:,1].tolist(),
    "edge_width": 0.001,
})

## For all mirrors, calculate position and rotation:
Tmirrors, _ = spherical_surface(R, mirrors_xyd[:,:2])

## Calculate intersection point for each mirror point
z_val = f + jnp.sqrt(f**2 + 2*Tmirrors[:,2]*(R-f))
p_orient = jnp.stack([jnp.zeros_like(z_val), jnp.zeros_like(z_val), z_val]).T

# Orient mirrors
Rmirrors = np.array(vmap(look_at_euler, in_axes= (0, 0))(Tmirrors, p_orient))
Rmirrors[:,2] = 0

## Define hexagon vertices
D = 1.2  # flat-to-flat distance in meters
R = D / np.sqrt(3)  # distance from center to vertex

# angles for vertices
angles = np.deg2rad(np.arange(0, 360, 60))

# coordinates of vertices
vertices = [[float(R * np.sin(a)), float(R * np.cos(a))] for a in angles]

## Add camera obstruction
config["obstructions"].append({
    "id": "Camera_Enclosure",
    "type": "box",
    "p1": [-1.45, -1.45, 15.5],
    "p2": [1.45, 1.45, 17.0],
})

## Add camera holding structure
mount_coords = [
    ([-1.65, -1.85], [1.65, -1.85]),
    ([-1.65, -1.85], [-1.65, 1.85]),
    ([1.65, -1.85], [1.65, 1.85]),
    ([-1.65, 1.85], [1.65, 1.85]),
]

for i, (p1_xy, p2_xy) in enumerate(mount_coords):
    config["obstructions"].append({
        "id": f"Camera_Mount_{i}",
        "type": "cylinder",
        "p1": p1_xy + [16.25],
        "p2": p2_xy + [16.25],
        "r": 0.15,
    })

## Add camera supports
for i, (x, y_sign) in enumerate([(x, y) for y in [-1, 1] for x in [-1.0, 0.0, 1.0]]):
    config["obstructions"].append({
        "id": f"Camera_Support_{i}",
        "type": "cylinder",
        "p1": [x, y_sign * 1.85, 16.25],
        "p2": [x, y_sign * 1.45, 16.25],
        "r": 0.15,
    })

## Add main masts:
R_struc = 6.3
struc_vertices = list(zip((R_struc * np.sin(angles)).tolist(),(R_struc * np.cos(angles)).tolist(), strict=False))

mast_coords = [
    ([1.65, 1.85, 16.25], [struc_vertices[1][0], struc_vertices[1][1], 1]),
    ([1.65, -1.85, 16.25], [struc_vertices[2][0], struc_vertices[2][1], 1]),
    ([-1.65, -1.85, 16.25], [struc_vertices[4][0], struc_vertices[4][1], 1]),
    ([-1.65, 1.85, 16.25], [struc_vertices[5][0], struc_vertices[5][1], 1]),
]

for i, (p1_xyz, p2_xyz) in enumerate(mast_coords):
    config["obstructions"].append({
        "id": f"Mast_{i}",
        "type": "cylinder",
        "p1": p1_xyz,
        "p2": p2_xyz,
        "r": 0.15,
    })

## Add trusses:
hook_cords = []
for i in range(4):
    hook_cords.append((np.array(mast_coords[i][0]) - np.array(mast_coords[i][1]))*2/3 + np.array(mast_coords[i][1]))

truss_coords = [
    (hook_cords[0]+np.array([-0.25,0,0]), [struc_vertices[0][0], struc_vertices[0][1], 1]),
    (hook_cords[3]+np.array([0.25,0,0]), [struc_vertices[0][0], struc_vertices[0][1], 1]),
    (hook_cords[1]+np.array([-0.25,0,0]), [struc_vertices[3][0], struc_vertices[3][1], 1]),
    (hook_cords[2]+np.array([0.25,0,0]), [struc_vertices[3][0], struc_vertices[3][1], 1]),
    (hook_cords[1], hook_cords[2]),
    (hook_cords[0], hook_cords[3]),
    (hook_cords[0]+np.array([0,-0.25,0]), [1.65, 0, 16.25]),
    (hook_cords[1]+np.array([0,0.25,0]), [1.65, 0, 16.25]),
    (hook_cords[2]+np.array([0,0.25,0]), [-1.65, 0, 16.25]),
    (hook_cords[3]+np.array([0,-0.25,0]), [-1.65, 0, 16.25]),
]

for i, (p1_xyz, p2_xyz) in enumerate(truss_coords):
    config["obstructions"].append({
        "id": f"Truss_{i}",
        "type": "cylinder",
        "p1": [float(x) for x in p1_xyz] if hasattr(p1_xyz, '__iter__') else p1_xyz,
        "p2": [float(x) for x in p2_xyz] if hasattr(p2_xyz, '__iter__') else p2_xyz,
        "r": 0.075,
    })

## Add wires:
wire_coords = [
    ([1.65, 1.85, 16.25], hook_cords[3]+np.array([0.25,0,0])),
    ([1.65, -1.85, 16.25], hook_cords[2]+np.array([0.25,0,0])),
    ([-1.65, -1.85, 16.25], hook_cords[1]+np.array([-0.25,0,0])),
    ([-1.65, 1.85, 16.25], hook_cords[0]+np.array([-0.25,0,0])),
    ([struc_vertices[1][0], struc_vertices[1][1], 1], hook_cords[3]+np.array([0.25,-0.2,0])),
    ([struc_vertices[2][0], struc_vertices[2][1], 1], hook_cords[2]+np.array([0.25,0.2,0])),
    ([struc_vertices[4][0], struc_vertices[4][1], 1], hook_cords[1]+np.array([-0.25,0.2,0])),
    ([struc_vertices[5][0], struc_vertices[5][1], 1], hook_cords[0]+np.array([-0.25,-0.2,0])),
]

for i, (p1_xyz, p2_xyz) in enumerate(wire_coords):
    config["obstructions"].append({
        "id": f"Tension_Wire_{i}",
        "type": "cylinder",
        "p1": [float(x) for x in p1_xyz] if hasattr(p1_xyz, '__iter__') else p1_xyz,
        "p2": [float(x) for x in p2_xyz] if hasattr(p2_xyz, '__iter__') else p2_xyz,
        "r": 0.0325,
    })

## Add connectors:
for i, hcoord in enumerate(hook_cords):
    config["obstructions"].append({
        "id": f"Connector_{i}",
        "type": "box",
        "p1": [float(x) for x in np.array([hcoord[0]*0.85, hcoord[1]*0.82, hcoord[2]-0.1])],
        "p2": [float(x) for x in np.array([hcoord[0]*1.07, hcoord[1]*1.1, hcoord[2]+0.1])],
    })

## Add mirrors one by one
for i in range(len(Tmirrors)):
    config["mirrors"].append({
        "id": f"M_{i}",
        "template": "spherical_32m",
        "position": [float(x) for x in Tmirrors[i]],
        "orientation": [float(x) for x in Rmirrors[i]],
        "aperture": {
            "type": "polygon",
            "vertices": vertices,
        },
        "stage": 0,
    })

# Save config to YAML file
with open('MST_North_like.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

In [6]:
telescope = Telescope.from_yaml('MST_North_like.yaml', MCIntegrator(256), key = jax.random.key(42))
scene = show_telescope(telescope)
scene.show(viewer='jupyter')

# MST South Like:

In [7]:
# Build config dictionary directly
config = {
    "telescope": {"name": "MST_South", "units": "m"},
    "mirror_templates": {
        "spherical_32m": {
            "surface": {
                "curvature": 1/32.14,
                "conic": 0.0,
                "aspheric": [],
            }
        }
    },
    "mirrors": [],
    "obstructions": [],
    "sensors": [],
}

# Focal length
f = 16
R = 19.2

# FlashCam sensor
config["sensors"].append({
    "id": "FlashCAM",
    "type": "hexagonal",
    "position": [0.0, 0.0, f],
    "orientation": [0.0, 0.0, 0.0],
    "centers_x": flashpix[:,0].tolist(),
    "centers_y": flashpix[:,1].tolist(),
    "edge_width": 0.001,
})

## For all mirrors, calculate position and rotation:
Tmirrors, _ = spherical_surface(R, mirrors_xyd[:,:2])

## Calculate intersection point for each mirror point
z_val = f + jnp.sqrt(f**2 + 2*Tmirrors[:,2]*(R-f))
p_orient = jnp.stack([jnp.zeros_like(z_val), jnp.zeros_like(z_val), z_val]).T

# Orient mirrors
Rmirrors = np.array(vmap(look_at_euler, in_axes= (0, 0))(Tmirrors, p_orient))
Rmirrors[:,2] = 0

## Define hexagon vertices
D = 1.2  # flat-to-flat distance in meters
R = D / np.sqrt(3)  # distance from center to vertex

# angles for vertices
angles = np.deg2rad(np.arange(0, 360, 60))

# coordinates of vertices
vertices = [[float(R * np.sin(a)), float(R * np.cos(a))] for a in angles]

## Add camera obstruction
config["obstructions"].append({
    "id": "Camera_Enclosure",
    "type": "box",
    "p1": [-1.5, -1.45, 15.7],
    "p2": [1.5, 1.45, 16.8],
})

## Add camera holding structure
mount_coords = [
    ([-1.65, -1.85], [1.65, -1.85]),
    ([-1.65, -1.85], [-1.65, 1.85]),
    ([1.65, -1.85], [1.65, 1.85]),
    ([-1.65, 1.85], [1.65, 1.85]),
]

for i, (p1_xy, p2_xy) in enumerate(mount_coords):
    config["obstructions"].append({
        "id": f"Camera_Mount_{i}",
        "type": "cylinder",
        "p1": p1_xy + [16.25],
        "p2": p2_xy + [16.25],
        "r": 0.15,
    })

## Add camera supports
for i, (x, y_sign) in enumerate([(x, y) for y in [-1, 1] for x in [-1.0, 0.0, 1.0]]):
    config["obstructions"].append({
        "id": f"Camera_Support_{i}",
        "type": "cylinder",
        "p1": [x, y_sign * 1.85, 16.25],
        "p2": [x, y_sign * 1.45, 16.25],
        "r": 0.15,
    })

## Add main masts:
R_struc = 6.3
struc_vertices = list(zip((R_struc * np.sin(angles)).tolist(),(R_struc * np.cos(angles)).tolist(), strict=False))

mast_coords = [
    ([1.65, 1.85, 16.25], [struc_vertices[1][0], struc_vertices[1][1], 1]),
    ([1.65, -1.85, 16.25], [struc_vertices[2][0], struc_vertices[2][1], 1]),
    ([-1.65, -1.85, 16.25], [struc_vertices[4][0], struc_vertices[4][1], 1]),
    ([-1.65, 1.85, 16.25], [struc_vertices[5][0], struc_vertices[5][1], 1]),
]

for i, (p1_xyz, p2_xyz) in enumerate(mast_coords):
    config["obstructions"].append({
        "id": f"Mast_{i}",
        "type": "cylinder",
        "p1": p1_xyz,
        "p2": p2_xyz,
        "r": 0.15,
    })

## Add trusses:
hook_cords = []
for i in range(4):
    hook_cords.append((np.array(mast_coords[i][0]) - np.array(mast_coords[i][1]))*2/3 + np.array(mast_coords[i][1]))

truss_coords = [
    (hook_cords[0]+np.array([-0.25,0,0]), [struc_vertices[0][0], struc_vertices[0][1], 1]),
    (hook_cords[3]+np.array([0.25,0,0]), [struc_vertices[0][0], struc_vertices[0][1], 1]),
    (hook_cords[1]+np.array([-0.25,0,0]), [struc_vertices[3][0], struc_vertices[3][1], 1]),
    (hook_cords[2]+np.array([0.25,0,0]), [struc_vertices[3][0], struc_vertices[3][1], 1]),
    (hook_cords[1], hook_cords[2]),
    (hook_cords[0], hook_cords[3]),
    (hook_cords[0]+np.array([0,-0.25,0]), [1.65, 0, 16.25]),
    (hook_cords[1]+np.array([0,0.25,0]), [1.65, 0, 16.25]),
    (hook_cords[2]+np.array([0,0.25,0]), [-1.65, 0, 16.25]),
    (hook_cords[3]+np.array([0,-0.25,0]), [-1.65, 0, 16.25]),
]

for i, (p1_xyz, p2_xyz) in enumerate(truss_coords):
    config["obstructions"].append({
        "id": f"Truss_{i}",
        "type": "cylinder",
        "p1": [float(x) for x in p1_xyz] if hasattr(p1_xyz, '__iter__') else p1_xyz,
        "p2": [float(x) for x in p2_xyz] if hasattr(p2_xyz, '__iter__') else p2_xyz,
        "r": 0.075,
    })

## Add wires:
wire_coords = [
    ([1.65, 1.85, 16.25], hook_cords[3]+np.array([0.25,0,0])),
    ([1.65, -1.85, 16.25], hook_cords[2]+np.array([0.25,0,0])),
    ([-1.65, -1.85, 16.25], hook_cords[1]+np.array([-0.25,0,0])),
    ([-1.65, 1.85, 16.25], hook_cords[0]+np.array([-0.25,0,0])),
    ([struc_vertices[1][0], struc_vertices[1][1], 1], hook_cords[3]+np.array([0.25,-0.2,0])),
    ([struc_vertices[2][0], struc_vertices[2][1], 1], hook_cords[2]+np.array([0.25,0.2,0])),
    ([struc_vertices[4][0], struc_vertices[4][1], 1], hook_cords[1]+np.array([-0.25,0.2,0])),
    ([struc_vertices[5][0], struc_vertices[5][1], 1], hook_cords[0]+np.array([-0.25,-0.2,0])),
]

for i, (p1_xyz, p2_xyz) in enumerate(wire_coords):
    config["obstructions"].append({
        "id": f"Tension_Wire_{i}",
        "type": "cylinder",
        "p1": [float(x) for x in p1_xyz] if hasattr(p1_xyz, '__iter__') else p1_xyz,
        "p2": [float(x) for x in p2_xyz] if hasattr(p2_xyz, '__iter__') else p2_xyz,
        "r": 0.0325,
    })

## Add connectors:
for i, hcoord in enumerate(hook_cords):
    config["obstructions"].append({
        "id": f"Connector_{i}",
        "type": "box",
        "p1": [float(x) for x in np.array([hcoord[0]*0.85, hcoord[1]*0.82, hcoord[2]-0.1])],
        "p2": [float(x) for x in np.array([hcoord[0]*1.07, hcoord[1]*1.1, hcoord[2]+0.1])],
    })

## Add mirrors one by one
for i in range(len(Tmirrors)):
    config["mirrors"].append({
        "id": f"M_{i}",
        "template": "spherical_32m",
        "position": [float(x) for x in Tmirrors[i]],
        "orientation": [float(x) for x in Rmirrors[i]],
        "aperture": {
            "type": "polygon",
            "vertices": vertices,
        },
        "stage": 0,
    })

# Save config to YAML file
with open('MST_South_like.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

In [8]:
telescope = Telescope.from_yaml('MST_South_like.yaml', MCIntegrator(32), key = jax.random.key(42))
scene = show_telescope(telescope)
scene.show(viewer='jupyter')